In [ ]:
from theia.test_data import build_fighter_jet_radar


build_fighter_jet_radar(0, 0, 0)

In [ ]:
from theia.test_data import build_single_target_from_Bodensee


traj = build_single_target_from_Bodensee()
traj

In [ ]:
import os

import folium
import shapely

from theia.coordinates import POSITIONS_OF_INTEREST
from theia.export_paraview import ParaviewExporter, PointOfInterest
from theia.grids import LatLonHeightGrid
from theia.mapping import RadarMap
from theia.terrain import elevationAt

out_dir = "output/paraview"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

grid = LatLonHeightGrid(
    lat_start=46.9204,
    lat_stop=48.0046,
    lat_res=0.005,
    lon_start=8.0854,
    lon_stop=9.4973,
    lon_res=0.005,
    height_start=3000,
    height_stop=3000,
    height_res=1,
)

exporter = ParaviewExporter(
    out_dir,
    grid.lat_start,
    grid.lat_stop,
    grid.lat_res,
    grid.lon_start,
    grid.lon_stop,
    grid.lon_res,
    # elevation_factor=5.0,
)

pois = [
    PointOfInterest(
        id=0,
        label="Rx",
        type="Rx",
        lat=46.9942,
        lon=8.5349,
        alt=elevationAt(46.9942, 8.5349) + 10,
    ),
    PointOfInterest(
        id=1,
        label="Rx Stallikon",
        type="Rx",
        lat=47.3258,
        lon= 8.4914,
        alt=elevationAt(47.3258,  8.4914) + 10,
    ),
    PointOfInterest(
        id=2,
        label="Rx Uetliberg",
        type="Rx",
        lat=POSITIONS_OF_INTEREST["Uetliberg"]["lat"],
        lon=POSITIONS_OF_INTEREST["Uetliberg"]["lon"],
        alt=POSITIONS_OF_INTEREST["Uetliberg"]["alt"],
    ),
    PointOfInterest(
        id=3,
        label="Rx ZH Airport",
        type="Rx",
        lat=47.4629,
        lon=8.5693,
        alt=elevationAt(47.4629, 8.5693),
    )
]

exporter.export([traj], pois)


map = RadarMap(
    trajectories={"Trajectory": traj},
    polygons={"roi": grid.get_bbox_polygon()},
).to_map()
for poi in pois:
    folium.Marker((poi.lat, poi.lon), tooltip=poi.label).add_to(map)
map

In [ ]:
from theia.coordinates import POSITIONS_OF_INTEREST
from theia.types import Point


center = Point(
    lat=POSITIONS_OF_INTEREST["Uetliberg"]["lat"],
    lon=POSITIONS_OF_INTEREST["Uetliberg"]["lon"],
    alt=POSITIONS_OF_INTEREST["Uetliberg"]["alt"],
)

start_point = Point(lat=47.2757, lon=8.3358, alt=1000)

In [ ]:
import datetime

import numpy as np
import shapely

from theia.maneuvers import EightManeuver

t_start = datetime.datetime(year=2026, month=4, day=28)

# maneuver = ConstantSpeedCurveManeuver(
#     center_point=center,
#     speed=300.0,
#     angular_arclength=-np.pi,
# )
maneuver = EightManeuver(
    center_point1=center,
    azimuth_center1_to_center2=np.pi / 2 / 3,
    elevation_center1_to_center2=0.0,
    speed=300.0,
)

times, points = maneuver.get_waypoints(start_time=t_start, start_pos=start_point)
curve = shapely.geometry.LineString([(p.lon, p.lat) for p in points + [points[-1]]])

In [ ]:
from theia.coordinates import CoordinateTransformations


p_center_ecef = np.array(
    CoordinateTransformations.geodetic_to_cartesian(
        center.lat,
        center.lon,
        center.alt,
    )
)
p_end_ecef = np.array(
    CoordinateTransformations.geodetic_to_cartesian(
        points[-1].lat,
        points[-1].lon,
        points[-1].alt,
    )
)
p_center2_ecef = 2 * p_end_ecef - p_center_ecef
lat, lon, alt = CoordinateTransformations.cartesian_to_geodetic(
    p_center2_ecef[0],
    p_center2_ecef[1],
    p_center2_ecef[2],
)
p_center2 = Point(lat=lat, lon=lon, alt=alt)

In [ ]:
timestamps = [t.timestamp() for t in times]
points_ecef = np.array(
    [CoordinateTransformations.geodetic_to_cartesian(*p.as_tuple()) for p in points]
)

In [ ]:
from matplotlib import pyplot as plt
from scipy.interpolate import CubicSpline

f_ecef = CubicSpline(timestamps, points_ecef, extrapolate=True)
v_ecef = f_ecef.derivative()
a_ecef = v_ecef.derivative()
t_eval = np.arange(timestamps[0], timestamps[-1], 0.1)
points_interpolated = f_ecef(t_eval)
speed_interpolated = np.linalg.norm(v_ecef(t_eval), axis=1)
accel_interpolated = np.linalg.norm(a_ecef(t_eval), axis=1)

fig, ax = plt.subplots()

ax.plot(t_eval, speed_interpolated)
# ax.set_ylim(299.995, 300.005)

In [ ]:
from theia.mapping import RadarMap
import folium


map = RadarMap().to_map()
map.location = (center.lat, center.lon)
folium.Marker((center.lat, center.lon), tooltip="Center").add_to(map)
# folium.Marker((47.34885858914191, 8.872821498768166), tooltip="Center 2").add_to(map)
folium.Marker((start_point.lat, start_point.lon), tooltip="Start point").add_to(map)
folium.GeoJson(curve).add_to(map)
map